In [6]:
from pathlib import Path

path_project_root = Path.cwd().parent
path_dataset = path_project_root.joinpath("data/proofwriter-dataset-V2020.12.3/OWA/depth-1")

In [7]:
import json
import random
import re

# Load the filtered subset (a JSON list of theories).
path_subset = path_project_root.joinpath(f"data/proofwriter-subset/meta-train.json")
with open(path_subset) as f:
    subset = json.load(f)

# Pick N_THEORIES theories at random. The same SEED always gives the same picks.
random.seed(42)
sample = random.sample(subset, 100)

path_sample = path_project_root.joinpath(f"data/proofwriter-subset/sample-train-seed42.json")
with open(path_sample, "w") as f:
    json.dump(sample, f, indent=2)

# Build one minimal data point per sampled theory:
# only the fact(s) and the rule of one proof, plus its two questions.
data_points = []

for theory in sample:
    # Both questions share the same proofs, so take them from the first one.
    first_question = list(theory["questions"].values())[0]
    proofs = first_question["proofs"][2:-2].split(" OR ")

    # Choose one proof at random. This runs after random.seed(42) above,
    # so the same proof is chosen every time the cell is run from scratch.
    proof = random.choice(proofs)

    # The names of the facts and the rule the proof uses,
    # e.g. ((triple8 triple6) -> rule3) gives triple8, triple6 and rule3.
    fact_names = re.findall(r"triple\d+", proof)
    rule_name = re.findall(r"rule\d+", proof)[0]

    # Facts in the order they appear in the original theory, then the rule.
    facts = {name: fact["text"] for name, fact in theory["triples"].items() if name in fact_names}
    rule = {rule_name: theory["rules"][rule_name]["text"]}

    data_points.append({
        # For tracing back: the theory id, the split it came from, and the chosen proof.
        "id": theory["id"],
        "split": "train",
        "proof": proof,
        # The premises: names point to the original triples and rules.
        "triples": facts,
        "rules": rule,
        # The new theory text: facts first, then the rule.
        "theory": " ".join(list(facts.values()) + list(rule.values())),
        # The True and False questions, unchanged, with their original names (e.g. Q3, Q4).
        "questions": theory["questions"],
    })

path_data_points = path_project_root.joinpath(f"data/proofwriter-subset/sample-train-seed42-no-clutter-single-proof.json")
with open(path_data_points, "w") as f:
    json.dump(data_points, f, indent=2)

In [8]:
import json

# Load the 100 data points (one minimal theory each).
path_data_points = path_project_root.joinpath("data/proofwriter-subset/sample-train-seed42-no-clutter-single-proof.json")
with open(path_data_points) as f:
    data_points = json.load(f)

# One document per data point in each list.
# Entry i of every list belongs to data point i.
fact_docs = []       # the fact(s) of the proof
rule_docs = []       # the rule of the proof
premise_docs = []    # facts and rule together
true_docs = []       # C: the True question
false_docs = []      # !C: the False question

for data_point in data_points:
    fact_docs.append(" ".join(data_point["triples"].values()))
    rule_docs.append(" ".join(data_point["rules"].values()))

    # The theory field is already the facts followed by the rule.
    premise_docs.append(data_point["theory"])

    # Each data point has exactly one True and one False question.
    for question in data_point["questions"].values():
        if question["answer"] is True:
            true_docs.append(question["question"])
        else:
            false_docs.append(question["question"])

# Every text, for fit to build the vocabulary from.
fit_docs = fact_docs + rule_docs + true_docs + false_docs

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

# ---- Binary: 1 if a word is present, 0 if not ----

binary_vectorizer = CountVectorizer(
    lowercase=True,                 # "Dave" and "dave" are the same word
    token_pattern=r"(?u)\b\w+\b",   # keep 1-letter words too (the default drops them)
    binary=True,
)

# Build the word list from every text.
binary_vectorizer.fit(fit_docs)

# Turn each list of documents into bags: one row per data point.
binary_fact_bags = binary_vectorizer.transform(fact_docs)
binary_rule_bags = binary_vectorizer.transform(rule_docs)
binary_premise_bags = binary_vectorizer.transform(premise_docs)
binary_true_bags = binary_vectorizer.transform(true_docs)
binary_false_bags = binary_vectorizer.transform(false_docs)

# Version 1: the prediction is the average of the facts bag and the rules bag.
binary_average_bags = (binary_fact_bags + binary_rule_bags) / 2


# ---- Counts: how many times each word appears ----

count_vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    binary=False,
)

count_vectorizer.fit(fit_docs)

count_fact_bags = count_vectorizer.transform(fact_docs)
count_rule_bags = count_vectorizer.transform(rule_docs)
count_premise_bags = count_vectorizer.transform(premise_docs)
count_true_bags = count_vectorizer.transform(true_docs)
count_false_bags = count_vectorizer.transform(false_docs)

count_average_bags = (count_fact_bags + count_rule_bags) / 2

In [ ]:
import random
from sklearn.metrics.pairwise import paired_cosine_distances

n_data_points = len(true_docs)


# ---- Q: a conclusion from a different, randomly chosen data point ----

# The seed makes the same Q get picked every time the cell is run.
random.seed(42)

q_docs = []
for i in range(n_data_points):
    # Choose one of the other data points (never data point i itself).
    other_data_points = [j for j in range(n_data_points) if j != i]
    other = random.choice(other_data_points)

    # Q is that data point's True or False conclusion, chosen at random.
    q_docs.append(random.choice([true_docs[other], false_docs[other]]))

binary_q_bags = binary_vectorizer.transform(q_docs)
count_q_bags = count_vectorizer.transform(q_docs)


# ---- Coverage: share of a statement's words that appear in the premises ----

# Both bags are binary: 1 if a word is present, 0 if not.
# Multiplying them word by word gives 1 only for words present in both,
# so the sum of that is the number of shared words.
# The sum of the statement bag is the number of words in the statement.
def coverage(premise_bag, statement_bag):
    shared_words = statement_bag.multiply(premise_bag).sum()
    statement_words = statement_bag.sum()
    return shared_words / statement_words


coverage_true = []
coverage_false = []
coverage_q = []

for i in range(n_data_points):
    premise_bag = binary_premise_bags[i]
    coverage_true.append(coverage(premise_bag, binary_true_bags[i]))
    coverage_false.append(coverage(premise_bag, binary_false_bags[i]))
    coverage_q.append(coverage(premise_bag, binary_q_bags[i]))

print("Coverage: index, id, C, !C, Q")
for i in range(n_data_points):
    print(
        i,
        data_points[i]["id"],
        round(coverage_true[i], 3),
        round(coverage_false[i], 3),
        round(coverage_q[i], 3),
    )


# ---- Cosine distance between the prediction and each statement ----

# paired_cosine_distances(A, B) compares row i of A only with row i of B,
# and returns one distance per row: 1 minus the cosine similarity.
# 0 means the two bags point in the same direction; 1 means they share no words.
def cosine_distances_per_data_point(prediction_bags, true_bags, false_bags, q_bags):
    distance_true = paired_cosine_distances(prediction_bags, true_bags)
    distance_false = paired_cosine_distances(prediction_bags, false_bags)
    distance_q = paired_cosine_distances(prediction_bags, q_bags)
    return distance_true, distance_false, distance_q


# For each version: the distances to C, to !C and to Q, one per data point.
cosine_results = {
    "Version 1, binary": cosine_distances_per_data_point(binary_average_bags, binary_true_bags, binary_false_bags, binary_q_bags),
    "Version 2, binary": cosine_distances_per_data_point(binary_premise_bags, binary_true_bags, binary_false_bags, binary_q_bags),
    "Version 1, counts": cosine_distances_per_data_point(count_average_bags, count_true_bags, count_false_bags, count_q_bags),
    "Version 2, counts": cosine_distances_per_data_point(count_premise_bags, count_true_bags, count_false_bags, count_q_bags),
}

for version, distances in cosine_results.items():
    distance_true, distance_false, distance_q = distances

    print()
    print(version + ": index, id, C, !C, Q, C and !C tied")
    for i in range(n_data_points):
        # A tie: C and !C are at exactly the same distance.
        tied = distance_true[i] == distance_false[i]
        print(
            i,
            data_points[i]["id"],
            round(distance_true[i], 3),
            round(distance_false[i], 3),
            round(distance_q[i], 3),
            tied,
        )

Coverage: index, id, C, !C, Q
0 AttNeg-OWA-D1-3180 1.0 0.75 0.5
1 RelNoneg-OWA-D1-2334 0.75 0.667 0.25
2 AttNoneg-OWA-D1-1366 1.0 0.75 0.333
3 RelNeg-OWA-D1-912 0.8 0.714 0.0
4 AttNeg-OWA-D1-1565 1.0 0.75 0.333
5 AttNeg-OWA-D1-2963 1.0 1.0 0.4
6 RelNoneg-OWA-D1-1187 1.0 0.8 0.333
7 RelNeg-OWA-D1-1859 0.8 1.0 0.0
8 RelNoneg-OWA-D1-1515 1.0 0.5 0.0
9 AttNeg-OWA-D1-2048 1.0 0.75 0.333
10 AttNoneg-OWA-D1-395 1.0 0.75 0.25
11 AttNeg-OWA-D1-18 1.0 0.75 0.167
12 AttNeg-OWA-D1-3005 1.0 0.75 0.2
13 AttNeg-OWA-D1-3100 1.0 0.75 0.333
14 AttNoneg-OWA-D1-3324 1.0 0.75 0.0
15 AttNoneg-OWA-D1-2055 1.0 0.75 0.25
16 AttNoneg-OWA-D1-1872 1.0 0.75 0.333
17 AttNoneg-OWA-D1-2270 1.0 0.75 0.2
18 AttNoneg-OWA-D1-2220 1.0 0.75 0.333
19 RelNeg-OWA-D1-345 1.0 0.833 0.333
20 AttNoneg-OWA-D1-1460 1.0 0.75 0.25
21 AttNeg-OWA-D1-706 1.0 0.75 0.25
22 RelNoneg-OWA-D1-2377 1.0 0.8 0.333
23 AttNeg-OWA-D1-2599 1.0 1.0 0.2
24 RelNeg-OWA-D1-1437 1.0 0.4 0.0
25 AttNoneg-OWA-D1-1057 1.0 0.75 0.25
26 RelNeg-OWA-D1-1181 1.0 0